In [1]:
# import shutil
# import sys
# from pathlib import Path
# import imageio_ffmpeg

# # 1. Đường dẫn file gốc bị sai tên
# src_file = Path(imageio_ffmpeg.get_ffmpeg_exe())

# # 2. Vị trí lưu mới (.venv\Scripts\ffmpeg.exe)
# target_file = Path(sys.executable).parent / "ffmpeg.exe"

# # 3. Sao chép và đổi tên thành ffmpeg.exe
# shutil.copy(src_file, target_file)

# print(f"Đã tạo thành công: {target_file}")

# # 4. Kiểm tra lại hệ thống đã nhận diện chưa
# print("Kiểm tra lại:", shutil.which("ffmpeg"))

In [2]:
import gc
import json
import torch
from transnetv2_pytorch import TransNetV2
from pathlib import Path

In [3]:
BASE_DIR = Path.cwd().parent / "L30_data"
VIDEO_DIR = BASE_DIR / "videos"
OUTPUT_DIR = BASE_DIR / "shots"
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".mpeg", ".mpg"}

MIN_SHOT_DURATION = 15.0
SKIP_EXISTING_OUTPUT = True

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransNetV2(device=device)
cpu_model = None

torch.set_grad_enabled(False)
print(f"Using device: {device}")

Using device: cuda


In [5]:
def to_seconds(val):
    """Chuyển đổi các định dạng thời gian về dạng float (giây)."""
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        if ":" in val:
            parts = val.split(":")
            if len(parts) == 3:
                return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
            elif len(parts) == 2:
                return float(parts[0]) * 60 + float(parts[1])
        return float(val)
    return float(val)


def smart_merge_shots(shots, min_duration):
    if not shots:
        return []

    # 1. Chuẩn hóa dữ liệu ban đầu
    cleaned_shots = []
    for s in shots:
        st = to_seconds(s["start_time"])
        et = to_seconds(s["end_time"])
        cleaned_shots.append({"start_time": st, "end_time": et, "duration": et - st})

    # 2. Vòng lặp gộp thông minh
    while True:
        if len(cleaned_shots) <= 1:
            break

        # Tìm shot có duration nhỏ nhất hiện tại
        min_shot_idx = min(
            range(len(cleaned_shots)), key=lambda i: cleaned_shots[i]["duration"]
        )
        min_shot = cleaned_shots[min_shot_idx]

        # Nếu shot nhỏ nhất đã đạt chuẩn min_duration -> Dừng
        if min_shot["duration"] >= min_duration:
            break

        # 3. So sánh 2 shot liền kề (trước và sau) để chọn shot ngắn hơn
        left_idx = min_shot_idx - 1 if min_shot_idx > 0 else None
        right_idx = (
            min_shot_idx + 1 if min_shot_idx < len(cleaned_shots) - 1 else None
        )

        # Quyết định gộp về bên trái hay bên phải
        if left_idx is not None and right_idx is not None:
            # Cả 2 bên đều tồn tại -> Chọn bên nào có duration nhỏ hơn
            if (
                cleaned_shots[left_idx]["duration"]
                <= cleaned_shots[right_idx]["duration"]
            ):
                target_idx = left_idx
            else:
                target_idx = right_idx
        elif left_idx is not None:
            target_idx = left_idx
        else:
            target_idx = right_idx

        # 4. Tiến hành gộp shot
        if target_idx < min_shot_idx:
            # Gộp vào shot bên trái
            cleaned_shots[target_idx]["end_time"] = min_shot["end_time"]
            cleaned_shots[target_idx]["duration"] = (
                cleaned_shots[target_idx]["end_time"]
                - cleaned_shots[target_idx]["start_time"]
            )
            cleaned_shots.pop(min_shot_idx)
        else:
            # Gộp vào shot bên phải
            cleaned_shots[target_idx]["start_time"] = min_shot["start_time"]
            cleaned_shots[target_idx]["duration"] = (
                cleaned_shots[target_idx]["end_time"]
                - cleaned_shots[target_idx]["start_time"]
            )
            cleaned_shots.pop(min_shot_idx)

    # 5. Format dữ liệu đầu ra
    final_shots = []
    for i, s in enumerate(cleaned_shots):
        final_shots.append(
            {
                "shot_id": i + 1,
                "start_time": round(s["start_time"], 2),
                "end_time": round(s["end_time"], 2),
                "duration": round(s["duration"], 2),
            }
        )

    return final_shots

In [6]:
if not VIDEO_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục video: {VIDEO_DIR}")

video_files = sorted(
    p for p in VIDEO_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
)

if not video_files:
    raise FileNotFoundError(
        f"Không tìm thấy file video nào trong {VIDEO_DIR} với đuôi {sorted(VIDEO_EXTENSIONS)}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processed_count = 0
skipped_count = 0
failed_count = 0

for idx, video_path in enumerate(video_files, start=1):
    relative_path = video_path.relative_to(VIDEO_DIR)
    output_file = (OUTPUT_DIR / relative_path).with_suffix(".json")
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if SKIP_EXISTING_OUTPUT and output_file.exists() and output_file.stat().st_size > 0:
        skipped_count += 1
        print(f"[{idx}/{len(video_files)}] SKIP {relative_path} (đã có output)")
        continue

    try:
        try:
            raw_shots = model.detect_scenes(video_path)
        except RuntimeError as e:
            is_cuda_oom = device == "cuda" and "out of memory" in str(e).lower()
            if not is_cuda_oom:
                raise

            print(f"[{idx}/{len(video_files)}] CUDA OOM với {relative_path}, chuyển sang CPU...")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            if cpu_model is None:
                cpu_model = TransNetV2(device="cpu")

            raw_shots = cpu_model.detect_scenes(video_path)

        final_shots = smart_merge_shots(raw_shots, min_duration=MIN_SHOT_DURATION)

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(final_shots, f, indent=2, ensure_ascii=False)

        processed_count += 1
        print(
            f"[{idx}/{len(video_files)}] DONE {relative_path} -> {output_file.relative_to(BASE_DIR)} "
            f"(raw: {len(raw_shots)}, merged: {len(final_shots)})"
        )

    except Exception as e:
        failed_count += 1
        print(f"[{idx}/{len(video_files)}] FAIL {relative_path}: {e}")

    finally:
        if "raw_shots" in locals():
            del raw_shots
        if "final_shots" in locals():
            del final_shots

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(
    f"\nHoàn tất. Tổng: {len(video_files)} | DONE: {processed_count} | "
    f"SKIP: {skipped_count} | FAIL: {failed_count}"
)
print(f"Output nằm trong: {OUTPUT_DIR}")

d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\torch\nn\modules\conv.py:720: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:217.)
  return F.conv3d(
d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\torch\nn\modules\linear.py:125: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 

[1/96] DONE L30_V001.mp4 -> shots\L30_V001.json (raw: 52, merged: 6)
[2/96] DONE L30_V002.mp4 -> shots\L30_V002.json (raw: 52, merged: 7)
[3/96] DONE L30_V003.mp4 -> shots\L30_V003.json (raw: 64, merged: 14)
[4/96] DONE L30_V004.mp4 -> shots\L30_V004.json (raw: 15, merged: 4)
[5/96] DONE L30_V005.mp4 -> shots\L30_V005.json (raw: 16, merged: 3)
[6/96] DONE L30_V006.mp4 -> shots\L30_V006.json (raw: 39, merged: 7)
[7/96] DONE L30_V007.mp4 -> shots\L30_V007.json (raw: 63, merged: 13)
[8/96] DONE L30_V008.mp4 -> shots\L30_V008.json (raw: 63, merged: 9)
[9/96] DONE L30_V009.mp4 -> shots\L30_V009.json (raw: 59, merged: 16)
[10/96] DONE L30_V010.mp4 -> shots\L30_V010.json (raw: 48, merged: 7)
[11/96] DONE L30_V011.mp4 -> shots\L30_V011.json (raw: 23, merged: 4)
[12/96] DONE L30_V012.mp4 -> shots\L30_V012.json (raw: 86, merged: 14)
[13/96] DONE L30_V013.mp4 -> shots\L30_V013.json (raw: 22, merged: 4)
[14/96] DONE L30_V014.mp4 -> shots\L30_V014.json (raw: 32, merged: 5)
[15/96] DONE L30_V015.mp4